<a href="https://colab.research.google.com/github/haider1272/-Butterfly-Image-Classification-using-YOLOv11/blob/main/VIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
import kagglehub
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,random_split
import torchvision
from torchvision.datasets import ImageFolder
from torchvision import transforms

In [ ]:
# Download latest version
path = kagglehub.dataset_download("muratkokludataset/rice-image-dataset")
print("Path to dataset files:", path)

In [ ]:
data_dir=os.path.join(path,'Rice_Image_Dataset')
embed_size=768
img_size=224
img_channels=3
patch_size=16
mha_heads=8
transformer_block_nums=4
mlp_modes=3072
learning_rate=0.001
epochs=10
batch_size = 32
num_patches=(img_size//patch_size)**2

In [ ]:
transform=transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor()
])

dataset=ImageFolder(data_dir,transform=transform)
classes=dataset.classes
num_classes=len(classes)
print(classes)

In [ ]:
train_size=int(0.7*(len(dataset)))
validation_size=int(0.2*(len(dataset)))
test_size=len(dataset)-train_size-validation_size

print(f"Train size: {train_size}")
print(f"Validation size: {validation_size}")
print(f"Test size): {test_size}")

In [ ]:
train_dataset,validation_dataset,test_dataset=random_split(dataset,[train_size,validation_size,test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(validation_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
#print few samples images from training data
plt.figure(figsize=(10,10))
for i, (images,labels) in enumerate(train_loader):
  if i==0:
    for j in range(10):
      plt.subplot(2,5,j+1)
      plt.imshow(images[j].permute(1,2,0))
      plt.title(classes[labels[j]])
      plt.axis('off')
    break
plt.show()



In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self):
      super().__init__()
      self.patch_embeddings=nn.Conv2d(img_channels,embed_size,kernel_size=patch_size,stride=patch_size)


    def forward(self,x):
      return self.patch_embeddings(x).flatten(2).transpose(1,2)




In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.layernorm1=nn.LayerNorm(embed_size)
    self.multi_attention_head=nn.MultiheadAttention(embed_size,mha_heads)
    self.layernorm2=nn.LayerNorm(embed_size)
    self.mlp=nn.Sequential(
        nn.Linear(embed_size,mlp_modes),
        nn.GELU(),
        nn.Linear(mlp_modes,embed_size)
    )

  def forward(self,x):
    residual_1=x
    x=self.layernorm1(x)
    x=self.multi_attention_head(x,x,x)[0]
    x=x+residual_1
    residual_2=x
    x=self.layernorm2(x)
    x=self.mlp(x)
    x=x+residual_2
    return x


In [ ]:
class MLP_Head(nn.Module):
  def __init__(self):
    super().__init__()
    self.layernorm=nn.LayerNorm(embed_size)
    self.mlp_head=nn.Sequential(
        nn.Linear(embed_size,num_classes)
    )
  def forward(self,x):
    x= x[:, 0, :]
    x=self.layernorm(x)
    return self.mlp_head(x)


In [ ]:
class VisionTransformer(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embedding=PatchEmbedding()
    self.cls_token=nn.Parameter(torch.randn(1,1,embed_size))
    self.pos_embedding=nn.Parameter(torch.randn(1,1+num_patches,embed_size))
    self.transformer_blocks=nn.Sequential(*[TransformerBlock() for _ in range(transformer_block_nums)])
    self.mlp_head=MLP_Head()

  def forward(self,x):
    x=self.patch_embedding(x)
    cls_token=self.cls_token.expand(x.shape[0],-1,-1)
    x=torch.cat((cls_token,x),dim=1)
    x=x+self.pos_embedding
    x=self.transformer_blocks(x)
    return self.mlp_head(x)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model=VisionTransformer().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)



In [ ]:
for epoch in range(epochs):

  train_loss = 0
  train_accuracy = 0
  val_accuracy = 0
  val_loss = 0

  model.train()

  for images,labels in train_loader:

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(outputs,labels)

    loss.backward()

    optimizer.step()

    train_loss += loss.item() * images.size(0)

    predicted = torch.argmax(outputs, dim=1)

    train_accuracy += (predicted == labels).sum().item()


  model.eval()

  with torch.no_grad():

    for images,labels in val_loader:

      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)

      loss = criterion(outputs,labels)

      val_loss += loss.item() * images.size(0)

      predicted = torch.argmax(outputs, dim=1)

      val_accuracy += (predicted == labels).sum().item()


  train_loss = train_loss / len(train_dataset)
  train_accuracy = train_accuracy / len(train_dataset)

  val_loss = val_loss / len(validation_dataset)
  val_accuracy = val_accuracy / len(validation_dataset)


  print(f"Epoch: {epoch+1}/{epochs}")
  print(f"Train Loss: {train_loss:.4f} Train Accuracy: {train_accuracy:.4f}")
  print(f"Validation Loss: {val_loss:.4f} Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
model.eval()

test_loss = 0
test_correct = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        test_correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


test_loss = test_loss / len(test_dataset)
test_accuracy = test_correct / len(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()